# First Nations Community Website Scraping

Scrape leadership and contact information from First Nations community websites.
Data flows: FN websites -> this notebook (CSV/JSON) -> NC source-manager import -> Minoo sync.

## Prerequisites
- NC source-manager running locally (port 8082) with communities seeded
- `pip install -e '.[notebooks]'` from indigenous-harvesters root

In [ ]:
import httpx
import pandas as pd
import json
import re
import time
from pathlib import Path
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse
from tqdm.notebook import tqdm
from dataclasses import dataclass, field, asdict
from datetime import datetime, timezone

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

NC_BASE_URL = "http://localhost:8082"
REQUEST_DELAY = 1.0  # seconds between scrape requests
REQUEST_TIMEOUT = 15.0

## 1. Fetch communities with websites from NC

In [ ]:
def fetch_communities_with_websites() -> list[dict]:
    """Fetch all communities that have a website URL from NC source-manager."""
    resp = httpx.get(f"{NC_BASE_URL}/api/v1/communities/with-source", timeout=30)
    resp.raise_for_status()
    communities = resp.json()["communities"]
    print(f"Fetched {len(communities)} communities with websites")
    return communities

communities = fetch_communities_with_websites()
df_communities = pd.DataFrame(communities)
df_communities[["id", "name", "website"]].head(10)

## 2. Page Discovery

For each community website, find leadership and contact pages.
Ported from NC's `crawler/internal/leadership/discovery.go`.

In [ ]:
# Page discovery patterns (ported from NC Go code)

LEADERSHIP_PATH_PATTERNS = [
    "chief-and-council", "chief-council", "chiefandcouncil",
    "leadership", "leaders", "council", "chief",
    "governance", "band-council", "elected-officials",
]

CONTACT_PATH_PATTERNS = [
    "contact", "contact-us", "contactus",
    "band-office", "bandoffice", "office",
    "location", "find-us", "get-in-touch",
]

LEADERSHIP_LINK_TEXTS = [
    "chief and council", "chief & council", "leadership",
    "council members", "elected officials", "governance",
    "band council", "chief",
]

CONTACT_LINK_TEXTS = [
    "contact", "contact us", "band office",
    "get in touch", "find us", "location",
]


@dataclass
class DiscoveredPage:
    url: str
    page_type: str  # 'leadership' or 'contact'


def classify_url(url: str) -> str | None:
    """Check if URL path matches leadership or contact patterns."""
    try:
        path = urlparse(url.lower()).path
    except Exception:
        return None
    for pattern in LEADERSHIP_PATH_PATTERNS:
        if pattern in path:
            return "leadership"
    for pattern in CONTACT_PATH_PATTERNS:
        if pattern in path:
            return "contact"
    return None


def classify_link_text(text: str) -> str | None:
    """Check if link text matches leadership or contact patterns."""
    lower = text.lower().strip()
    for pattern in LEADERSHIP_LINK_TEXTS:
        if pattern in lower:
            return "leadership"
    for pattern in CONTACT_LINK_TEXTS:
        if pattern in lower:
            return "contact"
    return None


def discover_pages(base_url: str, html: str) -> list[DiscoveredPage]:
    """Find leadership and contact pages from homepage links."""
    soup = BeautifulSoup(html, "lxml")
    seen: set[str] = set()
    pages: list[DiscoveredPage] = []

    for a_tag in soup.find_all("a", href=True):
        href = a_tag["href"]
        abs_url = urljoin(base_url, href)

        # Only follow links on same domain
        if urlparse(abs_url).netloc != urlparse(base_url).netloc:
            continue
        if abs_url in seen:
            continue

        # Try URL path classification first
        page_type = classify_url(abs_url)
        if page_type is None:
            # Fall back to link text
            link_text = a_tag.get_text(strip=True)
            page_type = classify_link_text(link_text)

        if page_type:
            seen.add(abs_url)
            pages.append(DiscoveredPage(url=abs_url, page_type=page_type))

    return pages

print("Page discovery functions loaded.")

## 3. Leadership Extraction

Extract chief, council, and other leadership from page text.
Ported from NC's `crawler/internal/leadership/leaders.go`.

In [ ]:
# Role patterns in descending priority (from NC Go code)
ROLE_PATTERNS = [
    (re.compile(r"(?i)\bdeputy\s+chief\b"), "deputy_chief"),
    (re.compile(r"(?i)\bchief\b"), "chief"),
    (re.compile(r"(?i)\bcouncill?ors?\b"), "councillor"),
    (re.compile(r"(?i)\bband\s+manager\b"), "band_manager"),
    (re.compile(r"(?i)\bexecutive\s+director\b"), "executive_director"),
    (re.compile(r"(?i)\bsecretary\b"), "secretary"),
    (re.compile(r"(?i)\btreasurer\b"), "treasurer"),
]

EMAIL_PATTERN = re.compile(r"[\w.+-]+@[\w.-]+\.\w{2,}")
PHONE_PATTERN = re.compile(r"\(?\d{3}\)?[-.\s]?\d{3}[-.\s]?\d{4}")
NAME_PATTERN = re.compile(r"^[A-Z][a-zA-Z'-]+(?:\s+[A-Z][a-zA-Z'-]+){1,4}$")
CONTACT_WINDOW = 5  # lines to scan after a name for contact info


@dataclass
class Person:
    name: str
    role: str
    email: str = ""
    phone: str = ""


def detect_role(line: str) -> str | None:
    for pattern, role in ROLE_PATTERNS:
        if pattern.search(line):
            return role
    return None


def looks_like_name(text: str) -> bool:
    text = text.strip()
    return 5 <= len(text) <= 50 and NAME_PATTERN.match(text) is not None


def clean_name(name: str) -> str:
    return name.strip().strip("\u2022\u00b7\u25cf\u2023\u25aa-\u2013\u2014:,.").strip()


def extract_name_from_role_line(line: str, role: str) -> str | None:
    """Extract name from a line like 'Chief John Smith'."""
    cleaned = line
    for pattern, _ in ROLE_PATTERNS:
        cleaned = pattern.sub("", cleaned)
    cleaned = cleaned.strip().strip(":-\u2013\u2014").strip()
    if looks_like_name(cleaned):
        return clean_name(cleaned)
    return None


def scan_contact_window(lines: list[str], start: int) -> tuple[str, str]:
    """Scan lines after a name for email and phone."""
    email = ""
    phone = ""
    end = min(start + CONTACT_WINDOW, len(lines))
    for wline in lines[start:end]:
        if not email:
            m = EMAIL_PATTERN.search(wline)
            if m:
                email = m.group()
        if not phone:
            m = PHONE_PATTERN.search(wline)
            if m:
                phone = m.group()
        if email and phone:
            break
    return email, phone


def extract_leaders(text: str) -> list[Person]:
    """Extract leadership from page text. Ported from NC Go ExtractLeaders."""
    lines = text.split("\n")
    leaders: list[Person] = []
    seen: set[str] = set()
    current_role: str | None = None

    for i, line in enumerate(lines):
        trimmed = line.strip()
        if not trimmed:
            continue

        role = detect_role(trimmed)
        if role:
            current_role = role
            name = extract_name_from_role_line(trimmed, role)
            if name and name not in seen:
                seen.add(name)
                email, phone = scan_contact_window(lines, i + 1)
                leaders.append(Person(name=name, role=role, email=email, phone=phone))
            continue

        if current_role and looks_like_name(trimmed):
            name = clean_name(trimmed)
            if name and name not in seen:
                seen.add(name)
                email, phone = scan_contact_window(lines, i + 1)
                leaders.append(Person(name=name, role=current_role, email=email, phone=phone))

    return leaders

print("Leadership extraction functions loaded.")

## 4. Contact Info Extraction

Extract band office contact details from contact pages.
Ported from NC's `crawler/internal/leadership/contact.go`.

In [ ]:
TOLL_FREE_PATTERN = re.compile(r"1[-.\s]?8\d{2}[-.\s]?\d{3}[-.\s]?\d{4}")
POSTAL_CODE_PATTERN = re.compile(r"[A-Z]\d[A-Z]\s?\d[A-Z]\d")
SOCIAL_MEDIA_PATTERNS = {
    "facebook": re.compile(r"https?://(?:www\.)?facebook\.com/[\w.-]+"),
    "twitter": re.compile(r"https?://(?:www\.)?(?:twitter|x)\.com/[\w.-]+"),
    "instagram": re.compile(r"https?://(?:www\.)?instagram\.com/[\w.-]+"),
    "youtube": re.compile(r"https?://(?:www\.)?youtube\.com/(?:@|channel/|c/)[\w.-]+"),
}


@dataclass
class ContactInfo:
    phone: str = ""
    toll_free: str = ""
    fax: str = ""
    email: str = ""
    postal_code: str = ""
    address: str = ""
    social_media: dict[str, str] = field(default_factory=dict)


def extract_phone_numbers(text: str) -> tuple[str, str, str]:
    """Categorize phone numbers into toll-free, main, and fax."""
    toll_free = ""
    phone = ""
    fax = ""
    for line in text.split("\n"):
        lower = line.lower()
        if not fax and "fax" in lower:
            m = PHONE_PATTERN.search(line)
            if m:
                fax = m.group()
                continue
        if not toll_free and TOLL_FREE_PATTERN.search(line):
            toll_free = TOLL_FREE_PATTERN.search(line).group()
            continue
        if not phone:
            m = PHONE_PATTERN.search(line)
            if m:
                phone = m.group()
    return toll_free, phone, fax


def extract_social_media(html: str) -> dict[str, str]:
    """Find social media links in page HTML."""
    found: dict[str, str] = {}
    for platform, pattern in SOCIAL_MEDIA_PATTERNS.items():
        m = pattern.search(html)
        if m:
            found[platform] = m.group()
    return found


def extract_contact(text: str, html: str = "") -> ContactInfo:
    """Extract contact info from page text. Ported from NC Go ExtractContact."""
    info = ContactInfo()

    m = EMAIL_PATTERN.search(text)
    if m:
        info.email = m.group()

    m = POSTAL_CODE_PATTERN.search(text.upper())
    if m:
        info.postal_code = m.group()

    info.toll_free, info.phone, info.fax = extract_phone_numbers(text)

    if html:
        info.social_media = extract_social_media(html)

    return info

print("Contact extraction functions loaded.")

## 5. Scrape Pipeline

For each community: fetch homepage -> discover pages -> scrape each -> extract data.

In [ ]:
@dataclass
class CommunityProfile:
    community_id: str
    community_name: str
    website: str
    leaders: list[Person] = field(default_factory=list)
    contact: ContactInfo | None = None
    pages_found: list[str] = field(default_factory=list)
    error: str = ""
    scraped_at: str = ""


def fetch_page(client: httpx.Client, url: str) -> tuple[str, str] | None:
    """Fetch a page, return (text, html) or None on error."""
    try:
        resp = client.get(url, timeout=REQUEST_TIMEOUT, follow_redirects=True)
        resp.raise_for_status()
        html = resp.text
        soup = BeautifulSoup(html, "lxml")
        # Remove script/style for text extraction
        for tag in soup(["script", "style", "nav", "footer", "header"]):
            tag.decompose()
        text = soup.get_text("\n", strip=True)
        return text, html
    except Exception as e:
        return None


def scrape_community(client: httpx.Client, community: dict) -> CommunityProfile:
    """Scrape a single community website for leadership and contact info."""
    profile = CommunityProfile(
        community_id=community["id"],
        community_name=community["name"],
        website=community.get("website", ""),
        scraped_at=datetime.now(timezone.utc).isoformat(),
    )

    if not profile.website:
        profile.error = "no_website"
        return profile

    # Ensure URL has scheme
    url = profile.website
    if not url.startswith("http"):
        url = "https://" + url

    # Fetch homepage
    result = fetch_page(client, url)
    if result is None:
        profile.error = "homepage_fetch_failed"
        return profile

    homepage_text, homepage_html = result

    # Discover leadership and contact pages
    pages = discover_pages(url, homepage_html)
    profile.pages_found = [f"{p.page_type}:{p.url}" for p in pages]

    # Also try extracting from homepage itself
    profile.leaders = extract_leaders(homepage_text)
    profile.contact = extract_contact(homepage_text, homepage_html)

    # Scrape discovered pages for richer data
    for page in pages:
        time.sleep(REQUEST_DELAY)
        page_result = fetch_page(client, page.url)
        if page_result is None:
            continue
        page_text, page_html = page_result

        if page.page_type == "leadership":
            page_leaders = extract_leaders(page_text)
            # Merge: prefer leadership page results (more complete)
            if page_leaders:
                existing_names = {p.name for p in profile.leaders}
                for leader in page_leaders:
                    if leader.name not in existing_names:
                        profile.leaders.append(leader)
                        existing_names.add(leader.name)

        elif page.page_type == "contact":
            page_contact = extract_contact(page_text, page_html)
            # Merge: fill in missing fields from contact page
            if profile.contact is None:
                profile.contact = page_contact
            else:
                for f in ["phone", "toll_free", "fax", "email", "postal_code", "address"]:
                    if not getattr(profile.contact, f) and getattr(page_contact, f):
                        setattr(profile.contact, f, getattr(page_contact, f))
                if page_contact.social_media:
                    profile.contact.social_media.update(page_contact.social_media)

    return profile

print("Scrape pipeline loaded.")

## 6. Run Scraper

Scrape all communities with websites. Uses a persistent HTTP client with polite delays.

In [ ]:
# Limit for testing — set to None to scrape all
LIMIT = 5

targets = communities[:LIMIT] if LIMIT else communities
print(f"Scraping {len(targets)} communities...")

profiles: list[CommunityProfile] = []

with httpx.Client(
    headers={"User-Agent": "MinooCommunityEnricher/0.1 (+https://minoo.app)"},
    follow_redirects=True,
) as client:
    for community in tqdm(targets, desc="Scraping"):
        profile = scrape_community(client, community)
        profiles.append(profile)
        if not profile.error:
            time.sleep(REQUEST_DELAY)

print(f"\nDone. {len(profiles)} communities scraped.")
print(f"  Errors: {sum(1 for p in profiles if p.error)}")
print(f"  With leaders: {sum(1 for p in profiles if p.leaders)}")
print(f"  With contact: {sum(1 for p in profiles if p.contact and (p.contact.phone or p.contact.email))}")

## 7. Assemble Dataset

In [ ]:
# Leaders table
leader_rows = []
for p in profiles:
    for leader in p.leaders:
        leader_rows.append({
            "community_id": p.community_id,
            "community_name": p.community_name,
            "name": leader.name,
            "role": leader.role,
            "email": leader.email,
            "phone": leader.phone,
            "source_url": p.website,
            "scraped_at": p.scraped_at,
        })

df_leaders = pd.DataFrame(leader_rows)
print(f"Leaders extracted: {len(df_leaders)}")
if not df_leaders.empty:
    print(f"Role distribution:\n{df_leaders['role'].value_counts()}")
    display(df_leaders.head(20))

In [ ]:
# Contact / band office table
contact_rows = []
for p in profiles:
    if p.contact is None:
        continue
    c = p.contact
    if not any([c.phone, c.toll_free, c.fax, c.email, c.postal_code, c.social_media]):
        continue
    contact_rows.append({
        "community_id": p.community_id,
        "community_name": p.community_name,
        "phone": c.phone,
        "toll_free": c.toll_free,
        "fax": c.fax,
        "email": c.email,
        "postal_code": c.postal_code,
        "address": c.address,
        "facebook": c.social_media.get("facebook", ""),
        "twitter": c.social_media.get("twitter", ""),
        "instagram": c.social_media.get("instagram", ""),
        "youtube": c.social_media.get("youtube", ""),
        "source_url": p.website,
        "scraped_at": p.scraped_at,
    })

df_contacts = pd.DataFrame(contact_rows)
print(f"Communities with contact info: {len(df_contacts)}")
if not df_contacts.empty:
    print(f"Fields populated:")
    for col in ["phone", "toll_free", "fax", "email", "postal_code", "facebook"]:
        count = df_contacts[col].astype(bool).sum()
        print(f"  {col}: {count}/{len(df_contacts)}")
    display(df_contacts.head(20))

In [ ]:
# Scrape summary table
summary_rows = []
for p in profiles:
    summary_rows.append({
        "community_id": p.community_id,
        "community_name": p.community_name,
        "website": p.website,
        "error": p.error,
        "leaders_found": len(p.leaders),
        "has_chief": any(l.role == "chief" for l in p.leaders),
        "has_phone": bool(p.contact and p.contact.phone),
        "has_email": bool(p.contact and p.contact.email),
        "pages_discovered": len(p.pages_found),
        "scraped_at": p.scraped_at,
    })

df_summary = pd.DataFrame(summary_rows)
print(f"\n=== SCRAPE SUMMARY ===")
print(f"Total communities: {len(df_summary)}")
print(f"Successful: {(~df_summary['error'].astype(bool)).sum()}")
print(f"With chief: {df_summary['has_chief'].sum()}")
print(f"With phone: {df_summary['has_phone'].sum()}")
print(f"With email: {df_summary['has_email'].sum()}")
display(df_summary)

## 8. Export Dataset

In [ ]:
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# CSV exports
if not df_leaders.empty:
    leaders_path = DATA_DIR / f"leaders_{timestamp}.csv"
    df_leaders.to_csv(leaders_path, index=False)
    print(f"Leaders: {leaders_path} ({len(df_leaders)} rows)")

if not df_contacts.empty:
    contacts_path = DATA_DIR / f"contacts_{timestamp}.csv"
    df_contacts.to_csv(contacts_path, index=False)
    print(f"Contacts: {contacts_path} ({len(df_contacts)} rows)")

summary_path = DATA_DIR / f"scrape_summary_{timestamp}.csv"
df_summary.to_csv(summary_path, index=False)
print(f"Summary: {summary_path} ({len(df_summary)} rows)")

# Full JSON export (for NC import later)
full_export = []
for p in profiles:
    entry = {
        "community_id": p.community_id,
        "community_name": p.community_name,
        "website": p.website,
        "error": p.error,
        "scraped_at": p.scraped_at,
        "leaders": [asdict(l) for l in p.leaders],
        "contact": asdict(p.contact) if p.contact else None,
        "pages_found": p.pages_found,
    }
    full_export.append(entry)

json_path = DATA_DIR / f"community_profiles_{timestamp}.json"
json_path.write_text(json.dumps(full_export, indent=2))
print(f"Full JSON: {json_path}")

## 9. Data Quality Review

Quick checks before this data is ready for NC import.

In [ ]:
if not df_leaders.empty:
    print("=== LEADER DATA QUALITY ===")
    # Check for duplicate names within same community
    dupes = df_leaders.groupby(["community_id", "name"]).size()
    dupes = dupes[dupes > 1]
    if not dupes.empty:
        print(f"\nWARN: {len(dupes)} duplicate name+community pairs:")
        print(dupes)
    else:
        print("No duplicate leaders within communities.")

    # Check for suspiciously short/long names
    short_names = df_leaders[df_leaders["name"].str.len() < 6]
    if not short_names.empty:
        print(f"\nWARN: {len(short_names)} suspiciously short names:")
        print(short_names[["community_name", "name", "role"]])

    # Communities with chief but no councillors
    communities_with_chief = set(df_leaders[df_leaders["role"] == "chief"]["community_id"])
    communities_with_council = set(df_leaders[df_leaders["role"] == "councillor"]["community_id"])
    chief_only = communities_with_chief - communities_with_council
    if chief_only:
        print(f"\nINFO: {len(chief_only)} communities have chief but no councillors extracted")

if not df_contacts.empty:
    print("\n=== CONTACT DATA QUALITY ===")
    # Validate email format
    if "email" in df_contacts.columns:
        bad_emails = df_contacts[
            df_contacts["email"].astype(bool) &
            ~df_contacts["email"].str.match(r"^[\w.+-]+@[\w.-]+\.\w{2,}$", na=False)
        ]
        if not bad_emails.empty:
            print(f"WARN: {len(bad_emails)} malformed emails")
            print(bad_emails[["community_name", "email"]])

    # Validate postal codes (Canadian format)
    if "postal_code" in df_contacts.columns:
        bad_pc = df_contacts[
            df_contacts["postal_code"].astype(bool) &
            ~df_contacts["postal_code"].str.match(r"^[A-Z]\d[A-Z]\s?\d[A-Z]\d$", na=False)
        ]
        if not bad_pc.empty:
            print(f"WARN: {len(bad_pc)} malformed postal codes")

print("\nData quality review complete.")